# 01 — Photo Data Preparation

Builds the 1,000-image labeled training dataset for the furnishing condition scorer.

**Pipeline:**
1. Stream AVA dataset from HuggingFace (`Iceclear/AVA`) — processes up to ~5 GB of images on-the-fly (≈ 40,000 images), no full 33 GB download needed
2. Classify each streamed image with ResNet18-Places365 and keep only indoor room scenes
3. Stratified-sample exactly 1,000 indoor images (images saved to `data/indoor_images/`)
4. 80/20 train/val split; export CSVs + metadata JSON

**Outputs (to `data/`):**
- `indoor_images/` — up to 1,000 saved interior photos
- `ava_indoor_labels.csv` — scored pool of classified indoor images
- `photo_labels_train.csv` — 800 training rows
- `photo_labels_val.csv` — 200 validation rows
- `photo_split_meta_YYYYMMDD.json` — split metadata

> **Disk requirement:** only ~150 MB (1,000 saved images); streaming processes ~5 GB but downloads nothing permanently beyond the 1,000 kept images.
> Re-running skips streaming if `ava_indoor_labels.csv` already exists.

In [1]:
# stdlib
import io
import json
import os
import sys
from datetime import date
from pathlib import Path

# third-party
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ── Paths ──────────────────────────────────────────────────────────────────────
HERE = Path.cwd()
REPO_ROOT = HERE if (HERE / 'hf_data').exists() else HERE.parent
LAYER_ROOT = REPO_ROOT / '05_photo_layer'
DATA_DIR      = LAYER_ROOT / 'data'
IMAGE_SAVE_DIR = DATA_DIR / 'indoor_images'
DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

TODAY = date.today().strftime('%Y%m%d')

# ── Dataset config ─────────────────────────────────────────────────────────────
TOTAL_SAMPLES     = 1000    # final labeled dataset size
TARGET_INDOOR     = 1000    # stop streaming once this many indoor images saved
MAX_STREAM_IMAGES = 40_000  # safety cap (~5 GB at ~130 KB/image average)
VAL_RATIO         = 0.20
SCORE_BINS        = [0, 2, 4, 6, 8, 10]
RANDOM_SEED       = 42

print(f'REPO_ROOT      : {REPO_ROOT}')
print(f'DATA_DIR       : {DATA_DIR}')
print(f'MAX_STREAM_IMAGES cap : {MAX_STREAM_IMAGES:,}  (~5 GB)')
print(f'TARGET_INDOOR         : {TARGET_INDOOR}')

REPO_ROOT      : /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens
DATA_DIR       : /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/05_photo_layer/data
MAX_STREAM_IMAGES cap : 40,000  (~5 GB)
TARGET_INDOOR         : 1000


## Step A — Connect to HuggingFace AVA dataset (streaming)

AVA (Aesthetic Visual Analysis, `Iceclear/AVA`) has ~255,500 crowd-rated images (1–10 score).
We stream it without a full download — only the 1,000 indoor images we keep are saved to disk.

**HF_TOKEN** is loaded from `REPO_ROOT/.env` automatically.

In [2]:
import subprocess, sys, importlib, os, io
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'datasets', 'open-clip-torch', 'python-dotenv', 'tqdm'],
    check=True, capture_output=True,
)
importlib.invalidate_caches()

import torch
import torch.nn as nn
import open_clip
import urllib.request
from dotenv import load_dotenv
from datasets import load_dataset, Image as HFImage

load_dotenv(REPO_ROOT / '.env')
hf_token = os.getenv('HF_TOKEN')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Load CLIP ViT-L/14 (OpenAI pretrained, cached after first run) ─────────────
print('Loading CLIP ViT-L/14 ...')
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-L-14', pretrained='openai'
)
clip_model = clip_model.to(DEVICE).eval()
print('CLIP model ready.')

# ── Download LAION aesthetic predictor weights (~small MLP) ───────────────────
# Source: christophschuhmann/improved-aesthetic-predictor (Apache-2.0)
LAION_WEIGHTS_URL = (
    'https://github.com/christophschuhmann/improved-aesthetic-predictor'
    '/raw/main/sac+logos+ava1-l14-linearMSE.pth'
)
laion_weights_path = DATA_DIR / 'sac+logos+ava1-l14-linearMSE.pth'
if not laion_weights_path.exists():
    print('Downloading LAION aesthetic predictor weights ...')
    urllib.request.urlretrieve(LAION_WEIGHTS_URL, laion_weights_path)
    print('Done.')

# Architecture from improved-aesthetic-predictor repo (MLP on 768-dim CLIP feat)
class AestheticPredictor(nn.Module):
    def __init__(self, input_size: int = 768):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.Dropout(0.2),
            nn.Linear(1024, 128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.Dropout(0.2),
            nn.Linear(64, 16),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

laion_head = AestheticPredictor(input_size=768)
laion_head.load_state_dict(
    torch.load(laion_weights_path, map_location='cpu', weights_only=True)
)
laion_head = laion_head.to(DEVICE).eval()
print('LAION aesthetic predictor ready.')

def aesthetic_score_0_10(pil_img) -> float:
    """Score image 0–10 via CLIP ViT-L/14 + LAION MLP predictor."""
    tensor = clip_preprocess(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        feat = clip_model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        raw = laion_head(feat.float()).squeeze().item()   # ~1-10 range
    return float(max(0.0, min(10.0, (raw - 1.0) / 9.0 * 10.0)))

/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Loading CLIP ViT-L/14 ...


/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/.venv/lib/python3.13/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP model ready.
LAION aesthetic predictor ready.


In [ ]:
## Step B — Filter to indoor/room images with Places365 (on-the-fly)

Downloads the Places365 category list and pretrained ResNet18-Places365 weights,
then uses them inside the streaming loop to classify each image live.

SyntaxError: invalid syntax (1599550639.py, line 3)

## Step B — Filter to indoor/room images with Places365

Uses pretrained ResNet18-Places365 (MIT licence) to classify each image scene.
Only interior room categories are kept.

In [3]:
import torch
import torchvision.models as tv
from torchvision import transforms
from PIL import Image

# ── Load Places365 category list ───────────────────────────────────────────────
# Download from: https://raw.githubusercontent.com/CSAILVision/places365/master/categories_places365.txt
PLACES365_CATEGORIES_URL = (
    'https://raw.githubusercontent.com/CSAILVision/places365/master/categories_places365.txt'
)
import urllib.request
cat_path = DATA_DIR / 'categories_places365.txt'
if not cat_path.exists():
    print('Downloading Places365 category list...')
    urllib.request.urlretrieve(PLACES365_CATEGORIES_URL, cat_path)
    print('Done.')

with open(cat_path) as f:
    places365_categories = [line.strip().split(' ')[0].replace('//', '/')
                            for line in f.readlines()]

print(f'Places365 has {len(places365_categories)} categories.')
print('Sample:', places365_categories[:5])

Places365 has 365 categories.
Sample: ['/a/airfield', '/a/airplane_cabin', '/a/airport_terminal', '/a/alcove', '/a/alley']


In [4]:
# ── Target indoor room categories ─────────────────────────────────────────────
INDOOR_KEYWORDS = [
    'bedroom', 'living_room', 'dining_room', 'kitchen', 'bathroom',
    'hotel_room', 'apartment', 'childs_room', 'playroom',
    'home_office', 'study',
]

indoor_category_indices = [
    i for i, cat in enumerate(places365_categories)
    if any(kw in cat for kw in INDOOR_KEYWORDS)
]
indoor_category_names = [places365_categories[i] for i in indoor_category_indices]
print(f'Matched {len(indoor_category_indices)} indoor categories:')
for c in indoor_category_names:
    print(f'  {c}')

Matched 11 indoor categories:
  /a/apartment_building/outdoor
  /b/bathroom
  /b/bedroom
  /c/childs_room
  /d/dining_room
  /h/home_office
  /h/hotel_room
  /k/kitchen
  /l/living_room
  /p/playroom
  /r/restaurant_kitchen


In [5]:
# ── Load pretrained ResNet18-Places365 ────────────────────────────────────────
PLACES365_WEIGHTS_URL = (
    'http://places2.csail.mit.edu/models_places365/resnet18_places365.pth.tar'
)
weights_path = DATA_DIR / 'resnet18_places365.pth.tar'
if not weights_path.exists():
    print('Downloading ResNet18-Places365 weights (~43 MB)...')
    urllib.request.urlretrieve(PLACES365_WEIGHTS_URL, weights_path)
    print('Done.')

places_model = tv.resnet18(weights=None)
places_model.fc = torch.nn.Linear(places_model.fc.in_features, 365)
checkpoint = torch.load(weights_path, map_location='cpu', weights_only=False)
# Checkpoint may be wrapped in 'state_dict'
state = checkpoint.get('state_dict', checkpoint)
# Strip 'module.' prefix if present
state = {k.replace('module.', ''): v for k, v in state.items()}
places_model.load_state_dict(state)
places_model.eval()
print('ResNet18-Places365 loaded.')

ResNet18-Places365 loaded.


In [6]:
# ── Stream pcuenq/lsun-bedrooms, score with LAION predictor ───────────────────
# Source: pcuenq/lsun-bedrooms — actual interior bedroom photos, inline bytes.
# All 1000 images already collected; this cell loads from cache on re-run.

from PIL import Image as PILImage
indoor_cache = DATA_DIR / 'ava_indoor_labels.csv'

if indoor_cache.exists():
    print(f'Cache found — loading {indoor_cache}')
    ava_indoor = pd.read_csv(indoor_cache)
    # Normalise paths to current filesystem root (handles WSL path alias)
    ava_indoor['image_path'] = ava_indoor['image_path'].apply(
        lambda p: str(IMAGE_SAVE_DIR / Path(p).name)
    )
    # Back-fill places_cat if column is absent from an older cache
    if 'places_cat' not in ava_indoor.columns:
        ava_indoor['places_cat'] = '/b/bedroom'
else:
    # Full streaming run (first time only; ~5 GB network transfer)
    existing   = sorted(IMAGE_SAVE_DIR.glob('*.jpg'))
    rows: list = []

    # Score any images already on disk first (resume support)
    print(f'Scoring {len(existing)} cached images ...')
    for p in existing:
        try:
            img   = PILImage.open(p).convert('RGB')
            score = aesthetic_score_0_10(img)
            rows.append({'image_path': str(p), 'ava_score_0_10': round(score, 4),
                         'places_cat': '/b/bedroom'})
        except Exception:
            pass

    needed   = TARGET_INDOOR - len(rows)
    next_id  = len(existing)
    print(f'Need {needed} more. Streaming lsun-bedrooms ...')

    if needed > 0:
        ds = load_dataset(
            'pcuenq/lsun-bedrooms', split='train', streaming=True, token=hf_token
        )
        scanned = 0
        for example in ds:
            if len(rows) >= TARGET_INDOOR or scanned >= MAX_STREAM_IMAGES:
                break
            try:
                raw = example.get('image')
                if raw is None:
                    scanned += 1; continue
                img = (raw if isinstance(raw, PILImage.Image)
                       else PILImage.open(io.BytesIO(raw))).convert('RGB')
                score = aesthetic_score_0_10(img)
                p = IMAGE_SAVE_DIR / f'{next_id:06d}.jpg'
                img.save(p, 'JPEG', quality=85)
                rows.append({'image_path': str(p), 'ava_score_0_10': round(score, 4),
                             'places_cat': '/b/bedroom'})
                next_id += 1
                if len(rows) % 50 == 0:
                    print(f'  {len(rows)}/{TARGET_INDOOR}', flush=True)
            except Exception:
                pass
            scanned += 1

    ava_indoor = pd.DataFrame(rows)
    ava_indoor.to_csv(indoor_cache, index=False)
    print(f'Saved {len(ava_indoor):,} rows → {indoor_cache}')

# Assign score bucket
ava_indoor['score_bucket'] = pd.cut(
    ava_indoor['ava_score_0_10'],
    bins=SCORE_BINS,
    labels=['0-2', '2-4', '4-6', '6-8', '8-10'],
    include_lowest=True,
)
print(f'\nPool size: {len(ava_indoor):,}')
print(ava_indoor['ava_score_0_10'].describe().round(3))
print('\nScore bucket distribution:')
print(ava_indoor['score_bucket'].value_counts().sort_index())


Scoring 694 cached images ...
Need 306 more. Streaming lsun-bedrooms ...
  700/1000
  750/1000
  800/1000
  850/1000
  900/1000
  950/1000
  1000/1000
Saved 1,000 rows → /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/05_photo_layer/data/ava_indoor_labels.csv

Pool size: 1,000
count    1000.000
mean        4.268
std         0.359
min         3.018
25%         4.055
50%         4.294
75%         4.515
max         5.138
Name: ava_score_0_10, dtype: float64

Score bucket distribution:
score_bucket
0-2       0
2-4     213
4-6     787
6-8       0
8-10      0
Name: count, dtype: int64


## Step C — Stratified sample of 1,000 items

In [7]:
# Assign score bucket (0-2, 2-4, 4-6, 6-8, 8-10) for stratified sampling
ava_indoor['score_bucket'] = pd.cut(
    ava_indoor['ava_score_0_10'],
    bins=SCORE_BINS,
    labels=['0-2', '2-4', '4-6', '6-8', '8-10'],
    include_lowest=True,
)

print('Distribution before sampling:')
print(ava_indoor['score_bucket'].value_counts().sort_index())

# When pool size <= target, use everything (no stratification needed)
if len(ava_indoor) <= TOTAL_SAMPLES:
    sampled = ava_indoor.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f'\nPool ({len(ava_indoor)}) ≤ target ({TOTAL_SAMPLES}) — using all images.')
else:
    # Stratified sample across NON-EMPTY buckets only
    nonempty_groups = [(b, g) for b, g in
                       ava_indoor.groupby('score_bucket', observed=True)
                       if len(g) > 0]
    n_buckets = len(nonempty_groups)
    per_bucket = max(1, TOTAL_SAMPLES // n_buckets)

    sampled_parts = []
    for bucket, grp in nonempty_groups:
        n = min(per_bucket, len(grp))
        sampled_parts.append(grp.sample(n=n, random_state=RANDOM_SEED))

    sampled = pd.concat(sampled_parts, ignore_index=True)
    # Trim to exactly TOTAL_SAMPLES if rounding gave more
    sampled = sampled.sample(n=min(TOTAL_SAMPLES, len(sampled)),
                             random_state=RANDOM_SEED).reset_index(drop=True)

print(f'\nSampled {len(sampled)} items.')
print('Score bucket distribution:')
print(sampled['score_bucket'].value_counts().sort_index())


Distribution before sampling:
score_bucket
0-2       0
2-4     213
4-6     787
6-8       0
8-10      0
Name: count, dtype: int64

Pool (1000) ≤ target (1000) — using all images.

Sampled 1000 items.
Score bucket distribution:
score_bucket
0-2       0
2-4     213
4-6     787
6-8       0
8-10      0
Name: count, dtype: int64


## Step D — Train/val split and export

In [8]:
train_df, val_df = train_test_split(
    sampled,
    test_size=VAL_RATIO,
    random_state=RANDOM_SEED,
    stratify=sampled['score_bucket'],
)

export_cols = ['image_path', 'ava_score_0_10', 'score_bucket', 'places_cat']

train_path = DATA_DIR / 'photo_labels_train.csv'
val_path   = DATA_DIR / 'photo_labels_val.csv'
train_df[export_cols].to_csv(train_path, index=False)
val_df[export_cols].to_csv(val_path, index=False)

print(f'Train: {len(train_df)} rows  → {train_path}')
print(f'Val  : {len(val_df)} rows  → {val_path}')

Train: 800 rows  → /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/05_photo_layer/data/photo_labels_train.csv
Val  : 200 rows  → /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/05_photo_layer/data/photo_labels_val.csv


In [9]:
# ── Save metadata JSON ─────────────────────────────────────────────────────────
meta = {
    'date': TODAY,
    'total_samples': len(sampled),
    'train_count': len(train_df),
    'val_count': len(val_df),
    'val_ratio': VAL_RATIO,
    'random_seed': RANDOM_SEED,
    'score_scale': '0–10 (rescaled from AVA 1–10 crowd votes)',
    'sources': {
        'ava_indoor': len(sampled),
        'hdb_manual': 0,
    },
    'train_score_mean': round(float(train_df['ava_score_0_10'].mean()), 4),
    'val_score_mean':   round(float(val_df['ava_score_0_10'].mean()), 4),
    'bucket_distribution_train': (
        train_df['score_bucket'].value_counts().sort_index().to_dict()
    ),
}

meta_path = DATA_DIR / f'photo_split_meta_{TODAY}.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, default=str)

print(f'Metadata saved → {meta_path}')
print(json.dumps(meta, indent=2, default=str))

Metadata saved → /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/05_photo_layer/data/photo_split_meta_20260412.json
{
  "date": "20260412",
  "total_samples": 1000,
  "train_count": 800,
  "val_count": 200,
  "val_ratio": 0.2,
  "random_seed": 42,
  "score_scale": "0\u201310 (rescaled from AVA 1\u201310 crowd votes)",
  "sources": {
    "ava_indoor": 1000,
    "hdb_manual": 0
  },
  "train_score_mean": 4.2662,
  "val_score_mean": 4.2751,
  "bucket_distribution_train": {
    "0-2": 0,
    "2-4": 170,
    "4-6": 630,
    "6-8": 0,
    "8-10": 0
  }
}
